<a href="https://colab.research.google.com/github/Abandonalo/SpatialGenUnity/blob/main/notebooks/Colab_ComfyUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# sanity check
import os
os.getcwd()

Mounted at /content/drive


'/content'

In [2]:
%cd /content/drive/MyDrive

import os
import subprocess
import sys

# Clone ComfyUI
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/drive/MyDrive/ComfyUI


!rm -f .git/index.lock

UPDATE_COMFY = False  # Set True only when you intentionally want to update ComfyUI.
if UPDATE_COMFY:
    subprocess.run(
        ["git", "stash", "push", "--include-untracked", "-m", "spatialgen-colab-autostash"],
        check=False,
    )
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    print("ℹ️ Skipping ComfyUI git pull for reproducible Colab startup")

# ----------------------------
# CUDA 12.6 check
# ----------------------------
def is_torch_cu126():
    try:
        import torch

        print("Found torch:", torch.__version__)
        print("CUDA build:", torch.version.cuda)

        return (
            torch.version.cuda == "12.6"
            and "+cu126" in torch.__version__
        )

    except Exception as e:
        print("Torch check failed:", e)
        return False


if is_torch_cu126():
    print("✅ CUDA 12.6 already installed — skipping")
else:
    print("⚠️ Installing CUDA 12.6 torch")

    !pip uninstall -y torch torchvision torchaudio

    !pip install \
        torch torchvision torchaudio \
        --index-url https://download.pytorch.org/whl/cu126

# Dependencies
!pip install -q -r requirements.txt
!pip install -q rembg onnxruntime
!pip install -q --upgrade "numpy>=2.3,<2.6" "trimesh>=4.5.0" "transformers==4.41.2" "tokenizers==0.19.1" "diffusers==0.29.2" "peft==0.10.0" sentencepiece



import torch
print("✅ CUDA available:", torch.cuda.is_available())
print("💡 GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("🔥 Torch CUDA:", torch.version.cuda)
print("📦 Torch:", torch.__version__)

/content/drive/MyDrive
/content/drive/MyDrive/ComfyUI
ℹ️ Skipping ComfyUI git pull for reproducible Colab startup
Found torch: 2.12.0+cu126
CUDA build: 12.6
✅ CUDA 12.6 already installed — skipping
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
✅ CUDA available: True
💡 GPU: Tesla T4
🔥 Torch CUDA: 12.6
📦 Torch: 2.12.0+cu126


In [3]:
import os
import re
import shutil
import subprocess
import tarfile
import urllib.request

# ============================================================
# Config
# ============================================================

ZROK_VERSION = "1.1.11"
ZROK_ENABLE_TOKEN = os.environ.get("ZROK_ENABLE_TOKEN", "").strip()
if not ZROK_ENABLE_TOKEN:
    try:
        from google.colab import userdata
        ZROK_ENABLE_TOKEN = (userdata.get("ZROK_ENABLE_TOKEN") or "").strip()
    except Exception:
        ZROK_ENABLE_TOKEN = ""
if not ZROK_ENABLE_TOKEN:
    import getpass
    ZROK_ENABLE_TOKEN = getpass.getpass("Enter zrok enable token: ").strip()
if not ZROK_ENABLE_TOKEN:
    raise RuntimeError("ZROK_ENABLE_TOKEN is required to create the Colab tunnel.")

archive_name = f"zrok_{ZROK_VERSION}_linux_amd64.tar.gz"
archive_url = (
    f"https://github.com/openziti/zrok/releases/download/"
    f"v{ZROK_VERSION}/{archive_name}"
)

extract_dir = "zrok_extract"

cache_dir = "/content/drive/MyDrive/.cache/zrok"
cache_binary = os.path.join(
    cache_dir,
    f"zrok_{ZROK_VERSION}"
)

local_binary = "./zrok"

os.makedirs(cache_dir, exist_ok=True)

# ============================================================
# Install / restore zrok binary
# ============================================================

if os.path.exists(cache_binary):
    print("✅ Reusing cached zrok binary from Drive")
    shutil.copy2(cache_binary, local_binary)

else:
    print("⬇️ Downloading zrok...")

    # Cleanup old artifacts
    for path in [archive_name, extract_dir]:
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)

    urllib.request.urlretrieve(
        archive_url,
        archive_name
    )

    print("📦 Extracting zrok...")

    os.makedirs(extract_dir, exist_ok=True)

    with tarfile.open(
        archive_name,
        "r:gz"
    ) as tar:
        tar.extractall(extract_dir)

    zrok_binary = None

    for root, _, files in os.walk(extract_dir):
        if "zrok" in files:
            zrok_binary = os.path.join(root, "zrok")
            break

    if zrok_binary is None:
        raise FileNotFoundError(
            "Could not locate zrok binary after extraction."
        )

    shutil.copy2(zrok_binary, local_binary)
    shutil.copy2(zrok_binary, cache_binary)

# Make executable
os.chmod(local_binary, 0o755)

if os.path.exists(cache_binary):
    os.chmod(cache_binary, 0o755)

# Verify zrok exists
subprocess.run([local_binary, "version"])

# ============================================================
# Enable zrok
# ============================================================

def enable_zrok(force=False):
    zrok_home = os.path.expanduser("~/.zrok")

    def reset_env():
        if os.path.exists(zrok_home):
            shutil.rmtree(zrok_home)

    # Optional clean reset
    if force:
        print("🧹 Resetting zrok environment...")
        reset_env()

    print("🔐 Validating zrok auth...")

    # --------------------------------------------------------
    # Try enable
    # --------------------------------------------------------

    result = subprocess.run(
        [local_binary, "enable", ZROK_ENABLE_TOKEN],
        capture_output=True,
        text=True,
    )

    combined = (
        (result.stdout or "")
        + "\n"
        + (result.stderr or "")
    ).strip()

    lowered = combined.lower()

    enable_ok = (
        result.returncode == 0
        or "already enabled" in lowered
        or "already have an enabled environment" in lowered
    )

    # --------------------------------------------------------
    # Broken Colab auth (/dev/tty) or stale state
    # --------------------------------------------------------

    if (
        "/dev/tty" in lowered
        or "no such device or address" in lowered
        or not enable_ok
    ):
        print(
            "⚠️ Broken or stale zrok auth detected — resetting"
        )

        reset_env()

        retry = subprocess.run(
            [local_binary, "enable", ZROK_ENABLE_TOKEN],
            capture_output=True,
            text=True,
        )

        retry_output = (
            (retry.stdout or "")
            + "\n"
            + (retry.stderr or "")
        ).strip()

        retry_lowered = retry_output.lower()

        enable_ok = (
            retry.returncode == 0
            or "already enabled" in retry_lowered
            or "already have an enabled environment"
            in retry_lowered
        )

        if not enable_ok:
            print("❌ zrok enable failed:")
            print(retry_output)
            return False

    # Do not create a throwaway `zrok share` here. The real reserved tunnel cell
    # below validates share permissions and retries auth if needed; doing it here
    # creates an extra public root/share in zrok.
    print("✅ zrok auth prepared; share validation will happen in the tunnel cell")
    return True



enable_zrok()

✅ Reusing cached zrok binary from Drive
🔐 Validating zrok auth...
✅ zrok auth prepared; share validation will happen in the tunnel cell


True

In [61]:
# ==========================================
# Hunyuan3D 2.1 + TripoSR + Essentials Custom Nodes
# ==========================================

import os
import re
import shutil
import sys
import subprocess
import urllib.request

COMFY_PATH = "/content/drive/MyDrive/ComfyUI"
CUSTOM_NODES = f"{COMFY_PATH}/custom_nodes"
HY3D_NODE = f"{CUSTOM_NODES}/ComfyUI-Hunyuan3d-2-1"
TRIPOSR_NODE = f"{CUSTOM_NODES}/ComfyUI-Flowty-TripoSR"
ESSENTIALS_NODE = f"{CUSTOM_NODES}/ComfyUI-essentials"
CHECKPOINT_DIR = f"{COMFY_PATH}/models/checkpoints"
TRIPOSR_MODEL_PATH = f"{CHECKPOINT_DIR}/TripoSRmodel.ckpt"
SD15_CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/v1-5-pruned-emaonly.safetensors"
INPAINT_CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/sd-v1-5-inpainting.ckpt"
CUSTOM_NODE_SETUP_STAMP = "/tmp/spatialgen_custom_nodes_setup_v2"
FORCE_CUSTOM_NODE_SETUP = False

print("🚀 Setting up ComfyUI custom nodes...")


def run(cmd):
    return subprocess.run(cmd, check=False)


def install_requirements_if_present(path, label):
    req_path = os.path.join(path, "requirements.txt")
    if os.path.exists(req_path):
        print(f"📦 Installing {label} dependencies...")
        run([sys.executable, "-m", "pip", "install", "-r", req_path])
    else:
        print(f"ℹ️ {label} has no requirements.txt")


def install_wheel_or_source(package_dir, label):
    dist_dir = os.path.join(package_dir, "dist")
    if os.path.exists(dist_dir):
        for file in os.listdir(dist_dir):
            if "linux" in file and file.endswith(".whl"):
                wheel_path = os.path.join(dist_dir, file)
                print(f"⚡ Installing {label} wheel: {file}")
                result = run([sys.executable, "-m", "pip", "install", wheel_path])
                if result.returncode == 0:
                    return
    print(f"🔧 Falling back to source build ({label})...")
    run([sys.executable, "-m", "pip", "install", package_dir])

def patch_triposr_numpy2_ptp(node_dir):
    """Patch old ndarray.ptp(...) calls for NumPy 2.x without downgrading NumPy globally."""
    if not os.path.isdir(node_dir):
        return

    ptp_pattern = re.compile(
        r"(?P<expr>\b[A-Za-z_][A-Za-z0-9_]*(?:\[[^\n\]]+\]|\.[A-Za-z_][A-Za-z0-9_]*)*)\.ptp\((?P<args>[^()\n]*)\)"
    )
    patched_files = []

    for root, _, files in os.walk(node_dir):
        for file_name in files:
            if not file_name.endswith(".py"):
                continue

            path = os.path.join(root, file_name)
            with open(path, "r", encoding="utf-8") as f:
                original = f.read()

            if ".ptp(" not in original:
                continue

            def replace_ptp(match):
                expr = match.group("expr")
                args = match.group("args").strip()
                return f"np.ptp({expr}{', ' + args if args else ''})"

            updated = ptp_pattern.sub(replace_ptp, original)
            if updated == original:
                continue

            if "import numpy as np" not in updated:
                updated = "import numpy as np\n" + updated

            with open(path, "w", encoding="utf-8") as f:
                f.write(updated)
            patched_files.append(os.path.relpath(path, node_dir))

    if patched_files:
        print("🩹 Patched Flowty TripoSR NumPy 2.x ptp usage:", ", ".join(patched_files))
    else:
        print("✅ Flowty TripoSR NumPy 2.x ptp patch not needed")


def patch_triposr_glb_export(node_dir):
    """Patch Flowty TripoSR viewer to honor a format input and export GLB."""
    init_path = os.path.join(node_dir, "__init__.py")
    if not os.path.exists(init_path):
        print("⚠️ Flowty TripoSR __init__.py not found; GLB export patch skipped")
        return

    with open(init_path, "r", encoding="utf-8") as f:
        original = f.read()

    if "SPATIALGEN_GLB_EXPORT_PATCH = True" in original:
        print("✅ Flowty TripoSR GLB export patch already applied")
        return

    viewer_class = '''class TripoSRViewer:
 SPATIALGEN_GLB_EXPORT_PATCH = True

 @classmethod
 def INPUT_TYPES(s):
  return {
   "required": {
    "mesh": ("MESH",),
    "format": (["obj", "glb"], {"default": "glb"})
   }
  }

 RETURN_TYPES = ()
 OUTPUT_NODE = True
 FUNCTION = "display"
 CATEGORY = "Flowty TripoSR"

 def display(self, mesh, format="glb"):
  saved = list()
  full_output_folder, filename, counter, subfolder, filename_prefix = get_save_image_path("meshsave",
  get_output_directory())
  filename_extension = str(format).lower()
  if filename_extension not in {"obj", "glb"}:
   filename_extension = "glb"

  transform = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, -1, 0, 0], [0, 0, 0, 1]])
  for (batch_number, single_mesh) in enumerate(mesh):
   filename_with_batch_num = filename.replace("%batch_num%", str(batch_number))
   file = f"{filename_with_batch_num}_{counter:05}_.{filename_extension}"
   mesh_to_export = single_mesh.copy()
   mesh_to_export.apply_transform(transform)
   mesh_to_export.export(path.join(full_output_folder, file), file_type=filename_extension)
   saved.append({
    "filename": file,
    "type": "output",
    "subfolder": subfolder
   })

  return {"ui": {"mesh": saved}}
'''

    pattern = r"class TripoSRViewer:\n.*?(?=\nNODE_CLASS_MAPPINGS\s*=)"
    updated, replacements = re.subn(pattern, viewer_class, original, count=1, flags=re.S)
    if replacements != 1:
        print("⚠️ Flowty TripoSR viewer class not found; GLB export patch skipped")
        return

    with open(init_path, "w", encoding="utf-8") as f:
        f.write(updated)
    print("🩹 Patched Flowty TripoSR viewer for GLB export")


def patch_hunyuan_trust_remote_code(node_dir):
    """Allow Hunyuan 2.1 PaintPBR custom UNet code to load with newer diffusers."""
    target_path = os.path.join(node_dir, "hy3dpaint", "utils", "multiview_utils.py")
    if not os.path.exists(target_path):
        print("⚠️ Hunyuan multiview_utils.py not found; trust_remote_code patch skipped")
        return False

    with open(target_path, "r", encoding="utf-8") as f:
        original = f.read()

    if "trust_remote_code=True" in original:
        print(f"✅ Hunyuan PaintPBR trust_remote_code patch already applied: {target_path}")
        return True

    pattern = r"(HunyuanPaintPipeline\.from_pretrained\(\s*model_path\s*,\s*torch_dtype\s*=\s*torch\.float16)(\s*,?\s*\))"
    updated, replacements = re.subn(
        pattern,
        r"\1,\n trust_remote_code=True\2",
        original,
        count=1,
        flags=re.S,
    )

    if replacements != 1:
        print("⚠️ Hunyuan PaintPBR loader shape changed; trust_remote_code patch skipped")
        print(f"   Checked: {target_path}")
        return False

    with open(target_path, "w", encoding="utf-8") as f:
        f.write(updated)

    with open(target_path, "r", encoding="utf-8") as f:
        verified = "trust_remote_code=True" in f.read()
    if verified:
        print(f"🩹 Patched Hunyuan PaintPBR loader with trust_remote_code=True: {target_path}")
    else:
        print(f"⚠️ Hunyuan PaintPBR patch write failed verification: {target_path}")
    return verified


if os.path.exists(CUSTOM_NODE_SETUP_STAMP) and not FORCE_CUSTOM_NODE_SETUP:
    print("✅ Reusing custom-node setup from this runtime")
else:
    # Install system build tools only if they are missing.
    if shutil.which("g++") and shutil.which("make"):
        print("✅ Build tools already available")
    else:
        print("🔧 Installing system build dependencies...")
        run(["apt-get", "update"])
        run(["apt-get", "install", "-y", "build-essential"])

    if not os.path.exists(HY3D_NODE):
        print("📦 Cloning Hunyuan3D 2.1...")
        run(["git", "clone", "https://github.com/visualbruno/ComfyUI-Hunyuan3d-2-1", HY3D_NODE])
    else:
        print("✅ Hunyuan3D already exists")
    patch_hunyuan_trust_remote_code(HY3D_NODE)

    if not os.path.exists(TRIPOSR_NODE):
        print("📦 Cloning Flowty TripoSR...")
        run(["git", "clone", "https://github.com/flowtyone/ComfyUI-Flowty-TripoSR", TRIPOSR_NODE])
    else:
        print("✅ Flowty TripoSR already exists")
    patch_triposr_glb_export(TRIPOSR_NODE)

    if not os.path.exists(ESSENTIALS_NODE):
        print("📦 Cloning ComfyUI Essentials...")
        run(["git", "clone", "https://github.com/cubiq/ComfyUI_essentials", ESSENTIALS_NODE])
    else:
        print("✅ Essentials already exists")

    install_requirements_if_present(HY3D_NODE, "Hunyuan")
    install_requirements_if_present(TRIPOSR_NODE, "TripoSR")
    install_requirements_if_present(ESSENTIALS_NODE, "Essentials")

    print("📦 Upgrading Trimesh for NumPy 2.x compatibility...")
    run([sys.executable, "-m", "pip", "install", "--upgrade", "trimesh>=4.5.0"])

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    if not os.path.exists(SD15_CHECKPOINT_PATH):
        print("⬇️ Downloading SD 1.5 checkpoint for TripoSR prompt image generation...")
        urllib.request.urlretrieve(
            "https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly.safetensors",
            SD15_CHECKPOINT_PATH,
        )
    else:
        print("✅ SD 1.5 checkpoint already exists")

    if not os.path.exists(INPAINT_CHECKPOINT_PATH):
        print("⬇️ Downloading SD 1.5 inpainting checkpoint for refinement...")
        urllib.request.urlretrieve(
            "https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt",
            INPAINT_CHECKPOINT_PATH,
        )
    else:
        print("✅ SD 1.5 inpainting checkpoint already exists")

    if not os.path.exists(TRIPOSR_MODEL_PATH):
        print("⬇️ Downloading TripoSR checkpoint...")
        urllib.request.urlretrieve(
            "https://huggingface.co/stabilityai/TripoSR/resolve/main/model.ckpt",
            TRIPOSR_MODEL_PATH,
        )
    else:
        print("✅ TripoSR checkpoint already exists")

    print("🧠 Installing custom rasterizer...")
    install_wheel_or_source(f"{HY3D_NODE}/hy3dpaint/custom_rasterizer", "rasterizer")

    print("🎨 Installing differentiable renderer...")
    install_wheel_or_source(f"{HY3D_NODE}/hy3dpaint/DifferentiableRenderer", "renderer")

    with open(CUSTOM_NODE_SETUP_STAMP, "w", encoding="utf-8") as f:
        f.write("ok")

    print("✅ All custom nodes installed successfully!")

# The setup stamp can skip the large install branch on warm runtimes; still
# ensure the refinement-specific checkpoint exists before starting the server.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
if not os.path.exists(INPAINT_CHECKPOINT_PATH):
    print("⬇️ Downloading SD 1.5 inpainting checkpoint for refinement...")
    urllib.request.urlretrieve(
        "https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt",
        INPAINT_CHECKPOINT_PATH,
    )
else:
    print("✅ SD 1.5 inpainting checkpoint already exists")

patch_hunyuan_trust_remote_code(HY3D_NODE)
patch_triposr_glb_export(TRIPOSR_NODE)
print("📦 Ensuring Trimesh is NumPy 2.x compatible...")
run([sys.executable, "-m", "pip", "install", "--upgrade", "trimesh>=4.5.0"])
patch_triposr_numpy2_ptp(TRIPOSR_NODE)

🚀 Setting up ComfyUI custom nodes...
✅ Reusing custom-node setup from this runtime
🩹 Patched Hunyuan PaintPBR loader with trust_remote_code=True: /content/drive/MyDrive/ComfyUI/custom_nodes/ComfyUI-Hunyuan3d-2-1/hy3dpaint/utils/multiview_utils.py
✅ Flowty TripoSR GLB export patch already applied
📦 Ensuring Trimesh is NumPy 2.x compatible...
✅ Flowty TripoSR NumPy 2.x ptp patch not needed


In [62]:
!pkill -f main.py

In [63]:
import os
import subprocess
import sys
import time
import torch

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "numpy>=2.3,<2.6",
        "trimesh>=4.5.0",
        "transformers>=4.46.0",
        "tokenizers>=0.20.0",
        "diffusers>=0.31.0",
        "peft>=0.13.0",
        "accelerate",
        "sentencepiece",
    ],
    check=True,
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Ensure PyTorch defaults to CUDA
if torch.cuda.is_available():
    torch.set_default_device("cuda")
    print("✅ Using GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ CUDA not available — something is wrong")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

DISABLE_SMART_MEMORY = False
FORCE_FP32 = False


subprocess.run(["pkill", "-f", "main.py"], check=False)
time.sleep(2)
COMFY_LOG_PATH = "/tmp/spatialgen_comfyui.log"
COMFY_DB_PATH = "/tmp/spatialgen_comfyui.db"
open(COMFY_LOG_PATH, "w", encoding="utf-8").close()


launch_args = [
    "python",
    "main.py",
    "--listen", "0.0.0.0",
    "--port", "8188",
    "--database-url", f"sqlite:///{COMFY_DB_PATH}",
]
if DISABLE_SMART_MEMORY:
    launch_args.append("--disable-smart-memory")
if FORCE_FP32:
    launch_args.append("--force-fp32")

comfy_log_handle = open(COMFY_LOG_PATH, "a", encoding="utf-8", buffering=1)
comfy_process = subprocess.Popen(
    launch_args,
    stdout=comfy_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

ready = False
log_pos = 0
for _ in range(180):
    with open(COMFY_LOG_PATH, "r", encoding="utf-8", errors="replace") as log_file:
        log_file.seek(log_pos)
        new_lines = log_file.readlines()
        log_pos = log_file.tell()

    for line in new_lines:
        print(line.strip())
        if "Starting server" in line or "To see the GUI go to" in line:
            ready = True

    if ready:
        break
    if comfy_process.poll() is not None:
        break

    time.sleep(0.5)

if ready:
    print("✅ ComfyUI is running on port 8188")
    print(f"🧾 ComfyUI logs: {COMFY_LOG_PATH}")
else:
    print("❌ ComfyUI may not have started correctly")
    print(f"🧾 Check ComfyUI logs: {COMFY_LOG_PATH}")

✅ Using GPU: Tesla T4
Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']}
Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'gemv_awq_w4a16', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'quantize_svdquant_w4a4', 'scaled_mm_nvfp4', 'scaled_mm_svdquant_w4a4', 'stochastic_rounding_fp8']}
Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_mxfp8', 'dequantize_nvfp4', 

In [64]:
from fastapi import FastAPI, HTTPException, Query, Request, Response
from starlette.concurrency import run_in_threadpool
import base64
import copy
import io
import json
import os
import random
import re
import requests
import time
import uuid

app = FastAPI()

COMFYUI_URL = "http://127.0.0.1:8188"
WORKFLOW_DIR = os.path.join(
    COMFY_PATH,
    "user",
    "default",
    "workflows",
)
FULL_WORKFLOW_TEMPLATE_PATH = os.path.join(WORKFLOW_DIR, "Full_Workflow_API.json")
UPLOAD_IMAGE_WORKFLOW_TEMPLATE_PATH = os.path.join(WORKFLOW_DIR, "Upload_Image_API.json")
TRIPOSR_WORKFLOW_TEMPLATE_PATH = os.path.join(WORKFLOW_DIR, "SpatialGen_TripoSR_API.json")
TRIPOSR_WORKFLOW_GITHUB_URL = "https://raw.githubusercontent.com/Abandonalo/SpatialGenUnity/main/tools/graphs/generation.json"
COMFY_INPUT_DIR = os.path.join(COMFY_PATH, "input")
UNITY_RUNS = {}
UNITY_RUN_ORDER = []
MAX_TRACKED_UNITY_RUNS = 25
COMFY_CLIENT_ID = "spatialgen-unity-colab-proxy"


def record_unity_run(request_id, **fields):
    if not request_id:
        return
    if request_id not in UNITY_RUNS:
        UNITY_RUN_ORDER.append(request_id)
        while len(UNITY_RUN_ORDER) > MAX_TRACKED_UNITY_RUNS:
            old_id = UNITY_RUN_ORDER.pop(0)
            UNITY_RUNS.pop(old_id, None)
        UNITY_RUNS[request_id] = {"request_id": request_id}
    UNITY_RUNS[request_id].update(fields)


def load_workflow_template(path):
    if not os.path.exists(path):
        return None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


TRIPOSR_WORKFLOW_TEMPLATE = {
    "3": {"inputs": {"seed": "__SEED__", "steps": "__STEPS__", "cfg": "__CFG__", "sampler_name": "euler", "scheduler": "normal", "denoise": 1, "model": ["4", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["5", 0]}, "class_type": "KSampler", "_meta": {"title": "KSampler"}},
    "4": {"inputs": {"ckpt_name": "__CHECKPOINT__"}, "class_type": "CheckpointLoaderSimple", "_meta": {"title": "Load Checkpoint"}},
    "5": {"inputs": {"width": 512, "height": 512, "batch_size": 1}, "class_type": "EmptyLatentImage", "_meta": {"title": "Empty Latent Image"}},
    "6": {"inputs": {"text": "__MESH_PROMPT__", "clip": ["4", 1]}, "class_type": "CLIPTextEncode", "_meta": {"title": "CLIP Text Encode (Mesh Source Prompt)"}},
    "7": {"inputs": {"text": "__MESH_NEG_PROMPT__", "clip": ["4", 1]}, "class_type": "CLIPTextEncode", "_meta": {"title": "CLIP Text Encode (Mesh Source Negative Prompt)"}},
    "8": {"inputs": {"samples": ["3", 0], "vae": ["4", 2]}, "class_type": "VAEDecode", "_meta": {"title": "VAE Decode"}},
    "43": {"inputs": {"filename_prefix": "spatialgen_mesh_source", "images": ["8", 0]}, "class_type": "SaveImage", "_meta": {"title": "Save Mesh Source Image"}},
    "50": {"inputs": {"upscale_method": "nearest-exact", "width": 512, "height": 512, "crop": "center", "image": ["8", 0]}, "class_type": "ImageScale", "_meta": {"title": "Upscale Image"}},
    "65": {"inputs": {"model": "u2net: general purpose", "providers": "CPU"}, "class_type": "RemBGSession+", "_meta": {"title": "RemBG Session"}},
    "64": {"inputs": {"rembg_session": ["65", 0], "image": ["50", 0]}, "class_type": "ImageRemoveBackground+", "_meta": {"title": "Image Remove Background"}},
    "53": {"inputs": {"mask": ["64", 1]}, "class_type": "MaskToImage", "_meta": {"title": "Mask To Image"}},
    "26": {"inputs": {"filename_prefix": "spatialgen_rmbg_source", "images": ["64", 0]}, "class_type": "SaveImage", "_meta": {"title": "Save RMBG Source Image"}},
    "27": {"inputs": {"filename_prefix": "spatialgen_rmbg_mask", "images": ["53", 0]}, "class_type": "SaveImage", "_meta": {"title": "Save RMBG Mask Image"}},
    "67": {"inputs": {"model": "__TRIPOSR_MODEL__", "chunk_size": 8192}, "class_type": "TripoSRModelLoader", "_meta": {"title": "TripoSR Model Loader"}},
    "66": {"inputs": {"geometry_resolution": "__GEOMETRY_RESOLUTION__", "threshold": "__TRIPOSR_THRESHOLD__", "model": ["67", 0], "reference_image": ["64", 0], "reference_mask": ["64", 1]}, "class_type": "TripoSRSampler", "_meta": {"title": "TripoSR Sampler"}},
    "68": {"inputs": {"format": "glb", "mesh": ["66", 0]}, "class_type": "TripoSRViewer", "_meta": {"title": "TripoSR Viewer"}}
}


TRIPOSR_WORKFLOW_UPLOAD_PATH = os.path.join(WORKFLOW_DIR, "generation.json")
TRIPOSR_WORKFLOW_ACTIVE_PATH = None


def ensure_triposr_workflow_template():
    global TRIPOSR_WORKFLOW_ACTIVE_PATH
    os.makedirs(WORKFLOW_DIR, exist_ok=True)

    if os.path.exists(TRIPOSR_WORKFLOW_UPLOAD_PATH):
        TRIPOSR_WORKFLOW_ACTIVE_PATH = TRIPOSR_WORKFLOW_UPLOAD_PATH
        print(f"📄 Using uploaded generation.json directly: {TRIPOSR_WORKFLOW_UPLOAD_PATH}")
        return

    TRIPOSR_WORKFLOW_ACTIVE_PATH = TRIPOSR_WORKFLOW_TEMPLATE_PATH
    try:
        print(f"⬇️ generation.json not found in workflows folder; downloading from GitHub: {TRIPOSR_WORKFLOW_GITHUB_URL}")
        response = requests.get(TRIPOSR_WORKFLOW_GITHUB_URL, timeout=30)
        response.raise_for_status()
        with open(TRIPOSR_WORKFLOW_TEMPLATE_PATH, "w", encoding="utf-8") as f:
            f.write(response.text)
        return
    except Exception as exc:
        print(f"⚠️ GitHub workflow download failed: {exc}")

    print(f"📝 Writing embedded TripoSR Unity workflow template to {TRIPOSR_WORKFLOW_TEMPLATE_PATH}")
    with open(TRIPOSR_WORKFLOW_TEMPLATE_PATH, "w", encoding="utf-8") as f:
        json.dump(TRIPOSR_WORKFLOW_TEMPLATE, f, indent=2)


def is_ui_workflow(workflow):
    return isinstance(workflow, dict) and "nodes" in workflow and "links" in workflow


def has_uploaded_asset_image(body):
    asset_image = body.get("asset_image")
    return isinstance(asset_image, dict) and bool(asset_image.get("image_base64"))


def uses_image_workflow(body):
    return str(body.get("input_mode", "")).lower() == "image" or has_uploaded_asset_image(body)


def get_colab_generation_model(body):
    value = str(body.get("colab_generation_model") or body.get("generation_model") or "").strip().lower()
    if value in {"tripo", "tripo_sr", "triposr"}:
        return "tripo_sr"
    return "hunyuan_2_1"


def load_selected_workflow(body):
    selected_model = get_colab_generation_model(body)
    if selected_model == "tripo_sr":
        if uses_image_workflow(body):
            raise HTTPException(
                status_code=400,
                detail="TripoSR Colab generation currently supports prompt-to-image-to-mesh only, not uploaded asset images.",
            )
        ensure_triposr_workflow_template()
        workflow_path = TRIPOSR_WORKFLOW_ACTIVE_PATH
    else:
        workflow_path = UPLOAD_IMAGE_WORKFLOW_TEMPLATE_PATH if uses_image_workflow(body) else FULL_WORKFLOW_TEMPLATE_PATH

    workflow = load_workflow_template(workflow_path)
    if workflow is None:
        raise HTTPException(
            status_code=400,
            detail=(
                "Workflow not found on Google Drive. Save "
                f"{os.path.basename(workflow_path)} to {WORKFLOW_DIR} before using /generate."
            ),
        )

    if is_ui_workflow(workflow):
        raise HTTPException(
            status_code=400,
            detail=(
                f"{os.path.basename(workflow_path)} is in ComfyUI UI format, not API format. "
                "Export/save the workflow in API format before using /generate."
            ),
        )

    return copy.deepcopy(workflow)


POSITIVE_PROMPT_NODE_IDS = {"55"}
NEGATIVE_PROMPT_NODE_IDS = {"56"}


def apply_prompt_inputs(workflow, prompt, negative_prompt=""):
    for node_id, node in workflow.items():
        if not isinstance(node, dict) or node.get("class_type") != "CLIPTextEncode":
            continue

        meta = node.get("_meta") or {}
        title = str(meta.get("title", "")).lower()
        inputs = node.setdefault("inputs", {})

        if str(node_id) in NEGATIVE_PROMPT_NODE_IDS:
            inputs["text"] = negative_prompt
        elif str(node_id) in POSITIVE_PROMPT_NODE_IDS:
            inputs["text"] = prompt
        elif "negative" in title:
            inputs["text"] = negative_prompt
        else:
            inputs["text"] = prompt


def apply_generation_inputs(workflow, generation):
    if not generation:
        return

    seed = generation.get("seed")
    steps = generation.get("steps")
    cfg = generation.get("cfg")
    sampler = generation.get("sampler")
    width = generation.get("width")
    height = generation.get("height")

    # Unity uses -1 to mean "pick a random seed"; ComfyUI validators require >= 0.
    if seed is None or int(seed) < 0:
        seed = random.randint(0, 2**31 - 1)

    for node in workflow.values():
        if not isinstance(node, dict):
            continue

        class_type = node.get("class_type")
        inputs = node.setdefault("inputs", {})

        if class_type == "KSampler":
            inputs["seed"] = int(seed)
            if steps is not None:
                inputs["steps"] = steps
            if cfg is not None:
                inputs["cfg"] = cfg
            if sampler:
                inputs["sampler_name"] = sampler
        elif class_type == "EmptyLatentImage":
            if width is not None:
                inputs["width"] = width
            if height is not None:
                inputs["height"] = height


def replace_placeholders(value, replacements):
    if isinstance(value, str):
        return replacements.get(value, value)
    if isinstance(value, list):
        return [replace_placeholders(item, replacements) for item in value]
    if isinstance(value, dict):
        return {key: replace_placeholders(item, replacements) for key, item in value.items()}
    return value


def resolve_triposr_model_name():
    model_name = os.environ.get("SPATIALGEN_TRIPO_MODEL", "TripoSRmodel.ckpt")
    model_path = os.path.join(COMFY_PATH, "models", "checkpoints", model_name)
    if not os.path.exists(model_path):
        raise HTTPException(
            status_code=400,
            detail=f"TripoSR model not found: {model_path}. Put TripoSRmodel.ckpt in ComfyUI/models/checkpoints.",
        )
    return model_name


def bind_triposr_workflow(workflow, body):
    generation = body.get("generation") or {}
    seed = generation.get("seed", -1)
    if seed is None or int(seed) < 0:
        seed = random.randint(0, 2**31 - 1)
    replacements = {
        "__SEED__": int(seed),
        "__STEPS__": int(generation.get("steps") or 30),
        "__CFG__": float(generation.get("cfg") or 7.0),
        "__CHECKPOINT__": os.environ.get("COMFY_CHECKPOINT", "v1-5-pruned-emaonly.safetensors"),
        "__MESH_PROMPT__": body.get("prompt") or body.get("positive_prompt") or "",
        "__MESH_NEG_PROMPT__": body.get("negative_prompt") or "",
        "__TRIPOSR_MODEL__": resolve_triposr_model_name(),
        "__GEOMETRY_RESOLUTION__": int(os.environ.get("SPATIALGEN_GEOMETRY_RESOLUTION", "256")),
        "__TRIPOSR_THRESHOLD__": float(os.environ.get("SPATIALGEN_TRIPO_THRESHOLD", "25")),
    }
    return replace_placeholders(workflow, replacements)


def sanitize_upload_filename(file_name, request_id):
    base_name = os.path.basename((file_name or "").strip())
    if not base_name:
        base_name = "proxy_image.png"

    stem, ext = os.path.splitext(base_name)
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._-") or "proxy_image"
    ext = ext.lower() if ext else ".png"
    if ext not in {".png", ".jpg", ".jpeg", ".webp"}:
        ext = ".png"

    return f"{request_id}_{stem}{ext}"


def persist_uploaded_asset_image(asset_image, request_id):
    if not isinstance(asset_image, dict):
        return None

    image_base64 = asset_image.get("image_base64") or ""
    if not image_base64:
        raise HTTPException(status_code=400, detail="Image workflow selected, but asset_image.image_base64 is empty.")

    try:
        image_bytes = base64.b64decode(image_base64, validate=True)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid asset_image.image_base64 payload: {exc}") from exc

    if not image_bytes:
        raise HTTPException(status_code=400, detail="Decoded asset image is empty.")

    os.makedirs(COMFY_INPUT_DIR, exist_ok=True)
    image_name = sanitize_upload_filename(asset_image.get("file_name"), request_id)
    image_path = os.path.join(COMFY_INPUT_DIR, image_name)
    with open(image_path, "wb") as f:
        f.write(image_bytes)
    return image_name


def apply_uploaded_image_inputs(workflow, image_name):
    if not image_name:
        return

    applied = False
    for node in workflow.values():
        if not isinstance(node, dict):
            continue

        class_type = node.get("class_type")
        if class_type not in {"Hy3D21LoadImageWithTransparency", "LoadImage"}:
            continue

        inputs = node.setdefault("inputs", {})
        inputs["image"] = image_name
        inputs["upload"] = "image"
        applied = True

    if not applied:
        raise HTTPException(
            status_code=400,
            detail="Upload_Image_API.json does not contain a supported image loader node.",
        )


def build_workflow(body):
    # In proxy mode the server owns workflow selection. Unity sends intent only.
    selected_model = get_colab_generation_model(body)
    workflow = load_selected_workflow(body)

    if selected_model == "tripo_sr":
        body["resolved_colab_generation_model"] = selected_model
        return bind_triposr_workflow(workflow, body)

    body["resolved_colab_generation_model"] = selected_model
    apply_prompt_inputs(
        workflow,
        body.get("prompt", ""),
        body.get("negative_prompt", ""),
    )
    apply_generation_inputs(workflow, body.get("generation") or {})

    if uses_image_workflow(body):
        image_name = persist_uploaded_asset_image(body.get("asset_image"), body.get("request_id") or str(uuid.uuid4()))
        apply_uploaded_image_inputs(workflow, image_name)

    return workflow


def fetch_history(prompt_id, timeout=30):
    response = requests.get(f"{COMFYUI_URL}/history/{prompt_id}", timeout=timeout)
    response.raise_for_status()
    return response.json()


def fetch_queue(timeout=30):
    response = requests.get(f"{COMFYUI_URL}/queue", timeout=timeout)
    response.raise_for_status()
    return response.json()


def get_history_entry(history_payload, prompt_id):
    if isinstance(history_payload, dict):
        if prompt_id in history_payload:
            return history_payload[prompt_id]
        if "outputs" in history_payload or "status" in history_payload:
            return history_payload
    return None


def get_queue_state(queue_payload, prompt_id):
    if find_queue_entry(queue_payload, prompt_id, "queue_running") is not None:
        return "running"
    if find_queue_entry(queue_payload, prompt_id, "queue_pending") is not None:
        return "pending"
    return "missing"


def find_queue_entry(queue_payload, prompt_id, key=None):
    if not isinstance(queue_payload, dict):
        return None
    keys = [key] if key else ["queue_running", "queue_pending"]
    needle = str(prompt_id)
    for queue_key in keys:
        entries = queue_payload.get(queue_key) or []
        if not isinstance(entries, list):
            continue
        for entry in entries:
            if needle in json.dumps(entry, default=str):
                return entry
    return None


def extract_prompt_from_queue_entry(entry):
    if isinstance(entry, list) and len(entry) >= 3 and isinstance(entry[2], dict):
        return entry[2]
    if isinstance(entry, dict):
        prompt = entry.get("prompt") or entry.get("workflow")
        if isinstance(prompt, dict):
            return prompt
    return {}


def summarize_node(node_id, node):
    if not isinstance(node, dict):
        return {"id": str(node_id), "class_type": "Unknown", "title": ""}
    meta = node.get("_meta") or {}
    inputs = node.get("inputs") or {}
    summary = {
        "id": str(node_id),
        "class_type": str(node.get("class_type", "Unknown")),
        "title": str(meta.get("title", "")),
    }
    for key in ("filename_prefix", "save_path", "file_name", "image", "model", "resolution", "octree_resolution"):
        if key in inputs and not isinstance(inputs.get(key), (list, dict)):
            summary[key] = inputs.get(key)
    return summary


def summarize_workflow(workflow, outputs_to_execute=None):
    if not isinstance(workflow, dict):
        return {"node_count": 0, "class_counts": {}, "output_nodes": [], "requested_output_nodes": []}
    class_counts = {}
    output_nodes = []
    requested_ids = {str(node_id) for node_id in (outputs_to_execute or [])}
    requested_output_nodes = []
    for node_id, node in workflow.items():
        if not isinstance(node, dict):
            continue
        class_type = str(node.get("class_type", "Unknown"))
        class_counts[class_type] = class_counts.get(class_type, 0) + 1
        lowered = class_type.lower()
        node_summary = summarize_node(node_id, node)
        if any(token in lowered for token in ("save", "export", "tripo", "hunyuan", "mesh", "glb")):
            output_nodes.append(node_summary)
        if str(node_id) in requested_ids:
            requested_output_nodes.append(node_summary)
    return {
        "node_count": len(workflow),
        "class_counts": dict(sorted(class_counts.items(), key=lambda item: item[0].lower())),
        "output_nodes": output_nodes[:40],
        "requested_output_nodes": requested_output_nodes[:40],
    }


def summarize_queue_entry(entry):
    if entry is None:
        return None
    workflow = extract_prompt_from_queue_entry(entry)
    if isinstance(entry, list):
        outputs_to_execute = entry[4] if len(entry) > 4 else []
        summary = summarize_workflow(workflow, outputs_to_execute)
        summary["prompt_number"] = entry[0] if len(entry) > 0 else None
        summary["prompt_id"] = entry[1] if len(entry) > 1 else ""
        summary["outputs_to_execute"] = outputs_to_execute
    elif isinstance(entry, dict):
        outputs_to_execute = entry.get("outputs_to_execute", [])
        summary = summarize_workflow(workflow, outputs_to_execute)
        summary["prompt_id"] = entry.get("prompt_id", "")
        summary["outputs_to_execute"] = outputs_to_execute
    else:
        summary = summarize_workflow(workflow)
    return summary


def extract_execution_error(history_entry):
    if not isinstance(history_entry, dict):
        return ""

    status = history_entry.get("status") or {}
    messages = status.get("messages") or []
    for message in messages:
        if not isinstance(message, list) or len(message) < 2:
            continue
        event_type, payload = message[0], message[1]
        if event_type != "execution_error" or not isinstance(payload, dict):
            continue
        return payload.get("exception_message") or payload.get("exception_type") or ""

    return ""


def extract_output_files(outputs):
    files = []
    seen = set()

    for node_id, node_output in outputs.items():
        if not isinstance(node_output, dict):
            continue

        for output_key, output_value in node_output.items():
            candidates = output_value if isinstance(output_value, list) else [output_value]
            for item in candidates:
                if not isinstance(item, dict) or "filename" not in item:
                    continue

                ref = {
                    "filename": item["filename"],
                    "subfolder": item.get("subfolder", ""),
                    "type": item.get("type", "output"),
                    "node_id": str(node_id),
                    "output_key": str(output_key),
                }
                dedupe_key = (ref["filename"], ref["subfolder"], ref["type"])
                if dedupe_key in seen:
                    continue
                seen.add(dedupe_key)
                files.append(ref)

    def sort_key(ref):
        ext = os.path.splitext(ref["filename"])[1].lower()
        if ext in {".glb", ".gltf", ".obj", ".fbx"}:
            return (0, ref["node_id"], ref["output_key"], ref["filename"])
        if ext in {".png", ".jpg", ".jpeg", ".webp"}:
            return (1, ref["node_id"], ref["output_key"], ref["filename"])
        return (2, ref["node_id"], ref["output_key"], ref["filename"])

    return sorted(files, key=sort_key)


def extract_images(files):
    return [
        ref for ref in files
        if os.path.splitext(ref["filename"])[1].lower() in {".png", ".jpg", ".jpeg", ".webp"}
    ]


def extract_meshes(files):
    return [
        ref for ref in files
        if os.path.splitext(ref["filename"])[1].lower() in {".glb", ".gltf", ".obj", ".fbx"}
    ]


@app.get("/health")
def health():
    try:
        response = requests.get(f"{COMFYUI_URL}/system_stats", timeout=10)
        response.raise_for_status()
        return {"status": "ok", "comfyui": "reachable"}
    except requests.RequestException as exc:
        raise HTTPException(status_code=503, detail=f"ComfyUI unreachable: {exc}") from exc


@app.post("/generate")
async def generate(req: Request):
    body = await req.json()

    request_id = body.get("request_id") or str(uuid.uuid4())
    body["request_id"] = request_id
    start_time = time.time()
    record_unity_run(
        request_id,
        status="building_workflow",
        started_at=start_time,
        prompt=(body.get("prompt") or body.get("positive_prompt") or "")[:240],
        input_mode=body.get("input_mode", ""),
        colab_generation_model=get_colab_generation_model(body),
    )

    try:
        workflow = build_workflow(body)
        record_unity_run(request_id, workflow_summary=summarize_workflow(workflow))
        response = requests.post(
            f"{COMFYUI_URL}/prompt",
            json={"prompt": workflow, "client_id": COMFY_CLIENT_ID},
            timeout=60,
        )

        if not response.ok:
            detail = response.text
            print(f"[{request_id}] ComfyUI /prompt validation failed: {detail}")
            record_unity_run(request_id, status="error", error=detail, failed_at=time.time())
            raise HTTPException(
                status_code=502,
                detail=f"ComfyUI /prompt rejected workflow: {detail}",
            )

        result = response.json()
        prompt_id = result.get("prompt_id")
        duration = time.time() - start_time

        if not prompt_id:
            record_unity_run(request_id, status="error", error=f"missing prompt_id: {result}", failed_at=time.time())
            raise HTTPException(status_code=502, detail=f"ComfyUI /prompt missing prompt_id: {result}")

        record_unity_run(
            request_id,
            status="queued",
            prompt_id=prompt_id,
            queued_at=time.time(),
            submit_duration=duration,
        )
        print(f"[{request_id}] queued prompt_id={prompt_id} ({duration:.2f}s)")

        return {
            "request_id": request_id,
            "status": "queued",
            "duration": duration,
            "prompt_id": prompt_id,
            "comfyui_response": result,
        }

    except HTTPException:
        raise
    except requests.RequestException as exc:
        duration = time.time() - start_time
        response_text = exc.response.text if exc.response is not None else str(exc)
        print(f"[{request_id}] request error: {response_text}")
        record_unity_run(request_id, status="error", error=response_text, failed_at=time.time())
        raise HTTPException(status_code=502, detail=f"Failed to submit prompt to ComfyUI: {response_text}") from exc
    except Exception as exc:
        duration = time.time() - start_time
        print(f"[{request_id}] unexpected error after {duration:.2f}s: {exc}")
        record_unity_run(request_id, status="error", error=str(exc), failed_at=time.time())
        raise HTTPException(status_code=500, detail=str(exc)) from exc


REFINEMENT_DEFAULT_NEGATIVE = "blur, low quality, noise, jpeg artifacts, distorted, deformed"
REFINEMENT_GRAPH_GITHUB_BASE = "https://raw.githubusercontent.com/Abandonalo/SpatialGenUnity/main/tools/graphs"
CANONICAL_VIEWS = ("Front", "Left", "Right", "Top")
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
MESH_EXTENSIONS = {".glb", ".gltf", ".obj", ".fbx"}
PLACEHOLDER_PATTERN = re.compile(r"__[A-Z0-9_]+__")

try:
    from PIL import Image
except Exception as exc:
    Image = None
    print(f"Pillow import failed; refinement endpoints will be unavailable: {exc}")


REFINEMENT_INPAINT_ONLY_TEMPLATE = {
    "1": {"inputs": {"image": "__RGB_IMAGE__"}, "class_type": "LoadImage"},
    "2": {"inputs": {"image": "__MASK_IMAGE__"}, "class_type": "LoadImage"},
    "3": {"inputs": {"seed": "__SEED__", "steps": "__STEPS__", "cfg": "__CFG__", "sampler_name": "dpmpp_2m", "scheduler": "karras", "denoise": "__DENOISE__", "model": ["4", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["9", 0]}, "class_type": "KSampler"},
    "4": {"inputs": {"ckpt_name": "__INPAINT_CHECKPOINT__"}, "class_type": "CheckpointLoaderSimple"},
    "6": {"inputs": {"text": "__POSITIVE_PROMPT__", "clip": ["4", 1]}, "class_type": "CLIPTextEncode"},
    "7": {"inputs": {"text": "__NEGATIVE_PROMPT__", "clip": ["4", 1]}, "class_type": "CLIPTextEncode"},
    "8": {"inputs": {"samples": ["3", 0], "vae": ["4", 2]}, "class_type": "VAEDecode"},
    "9": {"inputs": {"pixels": ["1", 0], "vae": ["4", 2], "mask": ["121", 0], "grow_mask_by": 24}, "class_type": "VAEEncodeForInpaint"},
    "12": {"inputs": {"image": ["2", 0], "channel": "red"}, "class_type": "ImageToMask"},
    "43": {"inputs": {"filename_prefix": "spatialgen_refined_tile", "images": ["8", 0]}, "class_type": "SaveImage"},
    "120": {"inputs": {"mask": ["12", 0], "value": 0.7}, "class_type": "ThresholdMask"},
    "121": {"inputs": {"mask": ["120", 0], "expand": 12, "tapered_corners": True}, "class_type": "GrowMask"},
}

REFINEMENT_TRIPO_FROM_RGB_TEMPLATE = {
    "1": {"inputs": {"image": "__RGB_IMAGE__"}, "class_type": "LoadImage"},
    "50": {"inputs": {"width": "__CROP_WIDTH__", "height": "__CROP_HEIGHT__", "x": "__CROP_X__", "y": "__CROP_Y__", "image": ["1", 0]}, "class_type": "ImageCrop"},
    "66": {"inputs": {"model": ["67", 0], "reference_image": ["50", 0], "reference_mask": ["123", 0], "geometry_resolution": "__GEOMETRY_RESOLUTION__", "threshold": "__TRIPOSR_THRESHOLD__"}, "class_type": "TripoSRSampler"},
    "67": {"inputs": {"model": "__TRIPOSR_MODEL__", "chunk_size": 8192}, "class_type": "TripoSRModelLoader"},
    "68": {"inputs": {"mesh": ["66", 0], "format": "glb"}, "class_type": "TripoSRViewer"},
    "122": {"inputs": {"image": "__TRIPO_SOLID_MASK__"}, "class_type": "LoadImage"},
    "123": {"inputs": {"image": ["122", 0], "channel": "red"}, "class_type": "ImageToMask"},
}


def require_pillow():
    if Image is None:
        raise RuntimeError("Pillow is required for refinement image composition in Colab.")


def default_tripo_model():
    return os.environ.get("SPATIALGEN_TRIPO_MODEL", "TripoSRmodel.ckpt")


def default_geometry_resolution():
    return int(os.environ.get("SPATIALGEN_GEOMETRY_RESOLUTION", "512"))


def default_tripo_threshold():
    return float(os.environ.get("SPATIALGEN_TRIPO_THRESHOLD", "25"))


def default_inpaint_checkpoint():
    return os.environ.get("COMFY_INPAINT_CHECKPOINT", "sd-v1-5-inpainting.ckpt")


def normalize_seed(seed):
    try:
        value = int(seed)
    except Exception:
        value = -1
    return value if value >= 0 else random.randint(0, 2**31 - 1)


def clamp_float(value, fallback, lo=None, hi=None):
    try:
        result = float(value)
    except Exception:
        result = float(fallback)
    if lo is not None:
        result = max(lo, result)
    if hi is not None:
        result = min(hi, result)
    return result


def clamp_int(value, fallback, lo=None):
    try:
        result = int(value)
    except Exception:
        result = int(fallback)
    if lo is not None:
        result = max(lo, result)
    return result


def load_refinement_graph_template(graph_name):
    upload_path = os.path.join(WORKFLOW_DIR, graph_name)
    workflow = load_workflow_template(upload_path)
    if workflow is not None:
        return copy.deepcopy(workflow)

    try:
        url = f"{REFINEMENT_GRAPH_GITHUB_BASE}/{graph_name}"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        workflow = response.json()
        os.makedirs(WORKFLOW_DIR, exist_ok=True)
        with open(upload_path, "w", encoding="utf-8") as f:
            json.dump(workflow, f, indent=2)
        print(f"Downloaded {graph_name} to {upload_path}")
        return workflow
    except Exception as exc:
        print(f"Falling back to embedded {graph_name}: {exc}")

    if graph_name == "refinement_inpaint_only.json":
        return copy.deepcopy(REFINEMENT_INPAINT_ONLY_TEMPLATE)
    if graph_name == "refinement_tripo_from_rgb.json":
        return copy.deepcopy(REFINEMENT_TRIPO_FROM_RGB_TEMPLATE)
    raise FileNotFoundError(f"Unknown refinement graph: {graph_name}")


def replace_graph_placeholders(value, replacements):
    if isinstance(value, dict):
        return {key: replace_graph_placeholders(item, replacements) for key, item in value.items()}
    if isinstance(value, list):
        return [replace_graph_placeholders(item, replacements) for item in value]
    if not isinstance(value, str):
        return value
    if value in replacements:
        return replacements[value]
    result = value
    for placeholder, replacement in replacements.items():
        if placeholder in result:
            result = result.replace(placeholder, str(replacement))
    return result


def collect_graph_placeholders(value):
    found = set()
    if isinstance(value, dict):
        for item in value.values():
            found.update(collect_graph_placeholders(item))
    elif isinstance(value, list):
        for item in value:
            found.update(collect_graph_placeholders(item))
    elif isinstance(value, str):
        found.update(PLACEHOLDER_PATTERN.findall(value))
    return found


def decode_base64_bytes(encoded, label):
    if not encoded:
        raise ValueError(f"Missing base64 image for {label}")
    raw = str(encoded).strip()
    if raw.startswith("data:") and ";base64," in raw:
        raw = raw.split(",", 1)[1]
    try:
        return base64.b64decode(raw, validate=True)
    except Exception as exc:
        raise ValueError(f"Invalid base64 image payload for {label}") from exc


def guess_image_extension(encoded):
    raw = str(encoded or "").strip().lower()
    if raw.startswith("data:image/jpeg") or raw.startswith("data:image/jpg"):
        return ".jpg"
    if raw.startswith("data:image/webp"):
        return ".webp"
    return ".png"


def write_refinement_input_image(encoded, label, run_token):
    os.makedirs(COMFY_INPUT_DIR, exist_ok=True)
    ext = guess_image_extension(encoded)
    path = os.path.join(COMFY_INPUT_DIR, f"{run_token}_{label}{ext}")
    with open(path, "wb") as f:
        f.write(decode_base64_bytes(encoded, label))
    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        raise RuntimeError(f"Failed to write ComfyUI input image: {path}")
    return os.path.basename(path)


def image_dimensions_from_base64(encoded):
    require_pillow()
    try:
        with Image.open(io.BytesIO(decode_base64_bytes(encoded, "dimensions"))) as im:
            return im.size
    except Exception:
        return None


def solid_white_png_base64(width, height):
    require_pillow()
    width = max(1, int(width))
    height = max(1, int(height))
    buffer = io.BytesIO()
    Image.new("RGB", (width, height), (255, 255, 255)).save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def tripo_crop_replacements(run_req):
    if run_req.get("crop_width") is not None and run_req.get("crop_height") is not None:
        crop_w = int(run_req["crop_width"])
        crop_h = int(run_req["crop_height"])
    else:
        crop_w, crop_h = image_dimensions_from_base64(run_req.get("rgb_image")) or (512, 512)
    return {
        "__CROP_WIDTH__": crop_w,
        "__CROP_HEIGHT__": crop_h,
        "__CROP_X__": int(run_req.get("crop_x") or 0),
        "__CROP_Y__": int(run_req.get("crop_y") or 0),
    }


def tripo_mask_dimensions(run_req):
    if run_req.get("crop_width") and run_req.get("crop_height"):
        return int(run_req["crop_width"]), int(run_req["crop_height"])
    return image_dimensions_from_base64(run_req.get("rgb_image")) or (512, 512)


def build_graph_replacements(run_req):
    replacements = {
        "__POSITIVE_PROMPT__": run_req.get("positive_prompt", ""),
        "__NEGATIVE_PROMPT__": run_req.get("negative_prompt", ""),
        "__MESH_PROMPT__": run_req.get("positive_prompt", ""),
        "__MESH_NEG_PROMPT__": run_req.get("negative_prompt", ""),
        "__SEED__": int(run_req.get("seed", 0)),
        "__STEPS__": int(run_req.get("steps", 1)),
        "__CFG__": float(run_req.get("cfg", 1.0)),
        "__DENOISE__": float(run_req.get("denoise", 1.0)),
        "__TRIPOSR_MODEL__": run_req.get("tripo_model") or default_tripo_model(),
        "__GEOMETRY_RESOLUTION__": int(run_req.get("geometry_resolution") or default_geometry_resolution()),
        "__TRIPOSR_THRESHOLD__": float(run_req.get("tripo_threshold") or default_tripo_threshold()),
        "__INPAINT_CHECKPOINT__": default_inpaint_checkpoint(),
        **tripo_crop_replacements(run_req),
    }

    run_token = uuid.uuid4().hex
    mode = run_req.get("mode")
    if mode in {"refine", "refine_inpaint_only"}:
        replacements["__RGB_IMAGE__"] = write_refinement_input_image(run_req.get("rgb_image"), "rgb", run_token)
        replacements["__MASK_IMAGE__"] = write_refinement_input_image(run_req.get("mask_image"), "mask", run_token)
        if run_req.get("depth_image"):
            replacements["__DEPTH_IMAGE__"] = write_refinement_input_image(run_req.get("depth_image"), "depth", run_token)
    elif mode == "tripo_from_rgb":
        mw, mh = tripo_mask_dimensions(run_req)
        replacements["__RGB_IMAGE__"] = write_refinement_input_image(run_req.get("rgb_image"), "rgb", run_token)
        replacements["__TRIPO_SOLID_MASK__"] = write_refinement_input_image(solid_white_png_base64(mw, mh), "tripo_solid_mask", run_token)
    return replacements


def inject_refinement_graph(graph_name, run_req):
    graph = load_refinement_graph_template(graph_name)
    injected = replace_graph_placeholders(graph, build_graph_replacements(run_req))
    unresolved = sorted(collect_graph_placeholders(injected))
    if unresolved:
        raise ValueError(f"Unresolved graph placeholders: {', '.join(unresolved)}")
    return injected


def wait_for_comfy_prompt(prompt_id):
    timeout_seconds = float(os.environ.get("COMFY_REFINE_TIMEOUT_SECONDS", "1200"))
    poll_interval = float(os.environ.get("COMFY_REFINE_POLL_INTERVAL", "1.0"))
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        history_payload = fetch_history(prompt_id)
        history_entry = get_history_entry(history_payload, prompt_id)
        if isinstance(history_entry, dict):
            error_message = extract_execution_error(history_entry)
            if error_message:
                raise RuntimeError(f"ComfyUI prompt {prompt_id} failed: {error_message}")
            if (history_entry.get("status") or {}).get("completed") or extract_output_files(history_entry.get("outputs") or {}):
                return history_entry
        time.sleep(poll_interval)
    raise TimeoutError(f"Timed out waiting for ComfyUI prompt {prompt_id}")


def download_output_ref_base64(ref):
    response = requests.get(
        f"{COMFYUI_URL}/view",
        params={"filename": ref["filename"], "subfolder": ref.get("subfolder", ""), "type": ref.get("type", "output")},
        timeout=120,
    )
    response.raise_for_status()
    return base64.b64encode(response.content).decode("ascii")


def send_refinement_graph_to_comfy(graph):
    response = requests.post(
        f"{COMFYUI_URL}/prompt",
        json={"prompt": graph, "client_id": COMFY_CLIENT_ID},
        timeout=60,
    )
    if not response.ok:
        raise RuntimeError(f"ComfyUI /prompt rejected refinement workflow: {response.text}")
    prompt_id = response.json().get("prompt_id")
    if not prompt_id:
        raise RuntimeError(f"ComfyUI /prompt missing prompt_id: {response.text}")

    history_entry = wait_for_comfy_prompt(prompt_id)
    files = extract_output_files(history_entry.get("outputs") or {})
    images = [ref for ref in files if os.path.splitext(ref["filename"])[1].lower() in IMAGE_EXTENSIONS]
    meshes = [ref for ref in files if os.path.splitext(ref["filename"])[1].lower() in MESH_EXTENSIONS]
    return {
        "prompt_id": prompt_id,
        "image_base64": download_output_ref_base64(images[0]) if images else "",
        "mesh_base64": download_output_ref_base64(meshes[0]) if meshes else "",
        "files": files,
    }


def canonicalize_view(view):
    key = str(view or "").strip().lower()
    for canonical in CANONICAL_VIEWS:
        if canonical.lower() == key:
            return canonical
    return "Front"


def validate_multi_view_request(body):
    views = body.get("views") or []
    if not views:
        raise ValueError("multi-view refinement requires at least one view")
    widths = {int(v.get("width") or 0) for v in views if int(v.get("width") or 0) > 0}
    heights = {int(v.get("height") or 0) for v in views if int(v.get("height") or 0) > 0}
    if len(widths) > 1 or len(heights) > 1:
        raise ValueError(f"multi-view refinement requires every view to share resolution (widths={sorted(widths)}, heights={sorted(heights)})")
    reconstruction_view = canonicalize_view(body.get("reconstructionView") or "Front")
    if not any(canonicalize_view(v.get("viewType")) == reconstruction_view for v in views):
        raise ValueError(f"reconstruction view '{reconstruction_view}' is missing from the request payload")
    for view in views:
        missing = [name for name in ("rgbBase64", "depthBase64", "maskBase64") if not view.get(name)]
        if missing:
            raise ValueError(f"view '{view.get('viewType', '')}' is missing: {', '.join(missing)}")


def effective_multi_view_prompt(prompt):
    prompt = str(prompt or "").strip()
    if not prompt:
        raise ValueError("positivePrompt is required for multi-view refinement")
    low = prompt.lower()
    cues = ["consistent colors across all four tiled views", "photorealistic sharp detail"]
    if "roof" in low:
        cues.append("same roof geometry visible from front sides and above")
    if "white" in low:
        cues.append("opaque bright white roof material not neutral gray filler")
    return f"{prompt}, " + ", ".join(cues)


def positive_for_inpaint_tile(base_positive, view_type):
    prompt = str(base_positive or "").strip()
    if not prompt:
        return prompt
    low = prompt.lower()
    view_type = canonicalize_view(view_type)
    extras = []
    if "roof" in low:
        if view_type == "Top":
            extras.append("strict top-down orthographic horizontal roof planes opaque bright white diffuse roofing")
        elif view_type in {"Left", "Right"}:
            extras.append("side elevation roof slope and eaves crisp white roofing material strong directional light")
        elif view_type == "Front":
            extras.append("front facade dominant roof mass bright white shingles clearly readable")
    return f"{prompt}, " + ", ".join(extras) if extras else prompt


def build_quadrant_map(reconstruction_view):
    reconstruction_view = canonicalize_view(reconstruction_view)
    ordered = [reconstruction_view] + [v for v in CANONICAL_VIEWS if v != reconstruction_view]
    return dict(zip(ordered, [(0, 0), (1, 0), (0, 1), (1, 1)]))


def decode_base64_image(data):
    require_pillow()
    return Image.open(io.BytesIO(decode_base64_bytes(data, "image")))


def composite_rgb_tiles(tile_b64_by_view, positions, cell_w, cell_h, fill=(0, 0, 0, 255)):
    require_pillow()
    canvas = Image.new("RGBA", (cell_w * 2, cell_h * 2), fill)
    for view_type, (col, row) in positions.items():
        payload = tile_b64_by_view.get(view_type)
        if not payload:
            continue
        tile = decode_base64_image(payload).convert("RGBA")
        if tile.size != (cell_w, cell_h):
            tile = tile.resize((cell_w, cell_h), Image.NEAREST)
        canvas.paste(tile, (col * cell_w, row * cell_h))
    buffer = io.BytesIO()
    canvas.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def run_inpaint_tile(view_payload, positive_prompt, negative_prompt, seed, steps, cfg, denoise):
    run_req = {
        "mode": "refine_inpaint_only",
        "positive_prompt": positive_prompt,
        "negative_prompt": negative_prompt,
        "rgb_image": view_payload.get("rgbBase64"),
        "depth_image": view_payload.get("depthBase64"),
        "mask_image": view_payload.get("maskBase64"),
        "seed": seed,
        "steps": max(1, int(steps)),
        "cfg": max(0.01, float(cfg)),
        "denoise": max(0.0, min(1.0, float(denoise))),
        "tripo_model": default_tripo_model(),
        "geometry_resolution": default_geometry_resolution(),
        "tripo_threshold": default_tripo_threshold(),
    }
    graph = inject_refinement_graph("refinement_inpaint_only.json", run_req)
    return send_refinement_graph_to_comfy(graph).get("image_base64") or ""


def run_tripo_from_rgb(rgb_image, crop_width=None, crop_height=None, crop_x=0, crop_y=0, seed=0):
    run_req = {
        "mode": "tripo_from_rgb",
        "positive_prompt": "refined spatial generation mesh",
        "negative_prompt": REFINEMENT_DEFAULT_NEGATIVE,
        "rgb_image": rgb_image,
        "seed": int(seed),
        "steps": 1,
        "cfg": 1.0,
        "denoise": 1.0,
        "tripo_model": default_tripo_model(),
        "geometry_resolution": default_geometry_resolution(),
        "tripo_threshold": max(18.0, default_tripo_threshold() - 3.0),
        "crop_width": crop_width,
        "crop_height": crop_height,
        "crop_x": crop_x,
        "crop_y": crop_y,
    }
    graph = inject_refinement_graph("refinement_tripo_from_rgb.json", run_req)
    return send_refinement_graph_to_comfy(graph).get("mesh_base64") or ""


def handle_colab_multi_view_refine(body):
    validate_multi_view_request(body)
    seed = normalize_seed(body.get("seed", -1))
    reconstruction_view = canonicalize_view(body.get("reconstructionView") or "Front")
    positions = build_quadrant_map(reconstruction_view)
    views = body.get("views") or []
    by_view = {canonicalize_view(v.get("viewType")): v for v in views}
    per_view_w = int(views[0].get("width") or 512)
    per_view_h = int(views[0].get("height") or 512)
    col, row = positions[reconstruction_view]
    positive = effective_multi_view_prompt(body.get("positivePrompt"))
    negative = str(body.get("negativePrompt") or "").strip() or REFINEMENT_DEFAULT_NEGATIVE
    steps = clamp_int(body.get("steps"), 20, lo=1)
    cfg = clamp_float(body.get("cfg"), 8.0, lo=0.01)
    denoise = clamp_float(body.get("denoise"), 0.3, lo=0.0, hi=1.0)

    refined_tiles = {}
    for index, view_type in enumerate(CANONICAL_VIEWS):
        view_payload = by_view.get(view_type)
        if view_payload is None:
            continue
        tile_seed = (seed + index * 1009 + ord(view_type[0])) & 0x7FFFFFFF
        refined_tiles[view_type] = run_inpaint_tile(
            view_payload,
            positive_for_inpaint_tile(positive, view_type),
            negative,
            tile_seed,
            steps,
            cfg,
            denoise,
        )

    rgb_refined_composite = composite_rgb_tiles(refined_tiles, positions, per_view_w, per_view_h)
    mesh_base64 = run_tripo_from_rgb(
        rgb_refined_composite,
        crop_width=per_view_w,
        crop_height=per_view_h,
        crop_x=col * per_view_w,
        crop_y=row * per_view_h,
        seed=seed,
    )
    return {
        "requestId": body.get("requestId", ""),
        "refinedViews": [{"viewType": reconstruction_view, "refinedImageBase64": rgb_refined_composite}],
        "meshBase64": mesh_base64,
        "success": True,
        "errorMessage": "",
    }


@app.post("/refine")
async def refine(req: Request):
    body = await req.json()
    try:
        return await run_in_threadpool(handle_colab_multi_view_refine, body)
    except Exception as exc:
        return {
            "requestId": body.get("requestId", ""),
            "refinedViews": [],
            "meshBase64": "",
            "success": False,
            "errorMessage": str(exc),
        }


@app.get("/result/{prompt_id}")
def get_result(prompt_id: str, request_timeout: float = 30):
    try:
        history_payload = fetch_history(prompt_id, timeout=request_timeout)
        history_entry = get_history_entry(history_payload, prompt_id)
        queue_payload = fetch_queue(timeout=request_timeout)
        queue_state = get_queue_state(queue_payload, prompt_id)
        queue_running_count = len(queue_payload.get("queue_running") or []) if isinstance(queue_payload, dict) else 0
        queue_pending_count = len(queue_payload.get("queue_pending") or []) if isinstance(queue_payload, dict) else 0
        queue_entry_summary = summarize_queue_entry(find_queue_entry(queue_payload, prompt_id))

        if history_entry is None:
            status = "running" if queue_state in {"running", "pending"} else "error"
            error_message = "Prompt is missing from both ComfyUI history and queue." if queue_state == "missing" else ""
            return {
                "status": status,
                "prompt_id": prompt_id,
                "completed": False,
                "exception_message": error_message,
                "queue_state": queue_state,
                "queue_running_count": queue_running_count,
                "queue_pending_count": queue_pending_count,
                "queue_entry": queue_entry_summary,
                "files": [],
                "meshes": [],
                "images": [],
                "history": {},
            }

        outputs = history_entry.get("outputs") or {}
        files = extract_output_files(outputs)
        meshes = extract_meshes(files)
        images = extract_images(files)
        completed = bool((history_entry.get("status") or {}).get("completed")) or bool(files)
        error_message = extract_execution_error(history_entry)
        status = "error" if error_message else ("success" if completed else "running")

        return {
            "status": status,
            "prompt_id": prompt_id,
            "completed": completed,
            "exception_message": error_message,
            "queue_state": queue_state,
            "queue_running_count": queue_running_count,
            "queue_pending_count": queue_pending_count,
            "queue_entry": queue_entry_summary,
            "files": files,
            "meshes": meshes,
            "images": images,
            "history": {prompt_id: history_entry},
        }
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to query ComfyUI history: {exc}") from exc


def summarize_unity_run(run, enrich=True, request_timeout=30):
    item = dict(run)
    started_at = item.get("started_at")
    if started_at:
        item["elapsed_seconds"] = round(time.time() - started_at, 1)

    prompt_id = item.get("prompt_id")
    if not enrich or not prompt_id:
        return item

    try:
        result = get_result(prompt_id, request_timeout=request_timeout)
        item["comfy_status"] = result.get("status", "unknown")
        item["completed"] = result.get("completed", False)
        item["queue_state"] = result.get("queue_state", "unknown")
        item["queue_running_count"] = result.get("queue_running_count", 0)
        item["queue_pending_count"] = result.get("queue_pending_count", 0)
        item["output_file_count"] = len(result.get("files", []))
        item["mesh_count"] = len(result.get("meshes", []))
        item["image_count"] = len(result.get("images", []))
        item["exception_message"] = result.get("exception_message", "")
        item["files"] = result.get("files", [])
        item["queue_entry"] = result.get("queue_entry")
        if result.get("exception_message"):
            item["status"] = "error"
        elif result.get("completed"):
            item["status"] = "completed"
        elif result.get("queue_state") in {"running", "pending"}:
            item["status"] = result.get("queue_state")
    except HTTPException as exc:
        item["status_lookup_error"] = str(exc.detail)
    except Exception as exc:
        item["status_lookup_error"] = str(exc)
    return item


@app.get("/unity_status")
def unity_status():
    ordered_ids = list(reversed(UNITY_RUN_ORDER))
    latest = summarize_unity_run(UNITY_RUNS[ordered_ids[0]], request_timeout=1.5) if ordered_ids else None
    runs = [summarize_unity_run(UNITY_RUNS[run_id], enrich=False) for run_id in ordered_ids]
    return {
        "run_count": len(runs),
        "latest": latest,
        "runs": runs,
    }


@app.get("/unity_status/{run_or_prompt_id}")
def unity_status_one(run_or_prompt_id: str):
    run = UNITY_RUNS.get(run_or_prompt_id)
    if run is None:
        for candidate in UNITY_RUNS.values():
            if candidate.get("prompt_id") == run_or_prompt_id:
                run = candidate
                break
    if run is None:
        raise HTTPException(status_code=404, detail="No Unity generation run matched that request_id or prompt_id.")
    return summarize_unity_run(run)


@app.get("/comfy_queue")
def comfy_queue():
    payload = fetch_queue()
    running = payload.get("queue_running") or [] if isinstance(payload, dict) else []
    pending = payload.get("queue_pending") or [] if isinstance(payload, dict) else []
    return {
        "running_count": len(running),
        "pending_count": len(pending),
        "running": [summarize_queue_entry(entry) for entry in running],
        "pending": [summarize_queue_entry(entry) for entry in pending],
    }


@app.get("/history/{prompt_id}")
def proxy_history(prompt_id: str):
    try:
        return fetch_history(prompt_id)
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to query ComfyUI history: {exc}") from exc


@app.get("/view")
def proxy_view(
    filename: str = Query(...),
    subfolder: str = Query(""),
    type: str = Query("output"),
):
    try:
        response = requests.get(
            f"{COMFYUI_URL}/view",
            params={
                "filename": filename,
                "subfolder": subfolder,
                "type": type,
            },
            timeout=60,
        )
        response.raise_for_status()
        return Response(
            content=response.content,
            media_type=response.headers.get("content-type", "application/octet-stream"),
        )
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to fetch ComfyUI output file: {exc}") from exc


In [65]:
import uvicorn
from threading import Thread

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = Thread(target=run)
thread.start()

print("✅ FastAPI proxy running on port 8000")


✅ FastAPI proxy running on port 8000


INFO:     Started server process [46942]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [66]:
import json
import re
import subprocess
import time

TARGET = "http://127.0.0.1:8000"
RESERVED_NAME = "comfyuitunnel"
RESERVED_ENDPOINT = f"https://{RESERVED_NAME}.share.zrok.io"


def extract_url(text):
    match = re.search(r"https://[^\s]+", text or "")
    return match.group(0) if match else None


def is_unauthorized(text):
    lowered = (text or "").lower()
    return "unauthorized" in lowered or "401" in lowered


def ensure_zrok_auth(force=False):
    print("🔑 Ensuring zrok auth is valid...")
    if enable_zrok(force=force):
        return True
    print("❌ zrok enable failed")
    return False


def reserve_zrok(allow_retry=True):
    print(f"🔄 Ensuring '{RESERVED_NAME}' is reserved...")
    res = subprocess.run(
        ["./zrok", "reserve", "public", TARGET, "-n", RESERVED_NAME],
        capture_output=True,
        text=True
    )
    if res.returncode == 0:
        print(f"✅ Reserved new share '{RESERVED_NAME}'")
    else:
        output = f"{res.stdout.strip()} {res.stderr.strip()}"
        print(f"ℹ️ Reserve command output (expected if already reserved): {output}")
        if allow_retry and is_unauthorized(output):
            print("⚠️ zrok reserve reported unauthorized. Re-enabling and retrying once...")
            if ensure_zrok_auth(force=True):
                reserve_zrok(allow_retry=False)


def start_reserved_zrok():
    return subprocess.Popen(
        ["./zrok", "share", "reserved", RESERVED_NAME, "--headless"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )


def stop_existing_unity_tunnel():
    global unity_zrok_process
    try:
        if unity_zrok_process and unity_zrok_process.poll() is None:
            print("♻️ Stopping previous Unity zrok tunnel process from this notebook...")
            unity_zrok_process.terminate()
            try:
                unity_zrok_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                unity_zrok_process.kill()
    except NameError:
        pass


def start_and_monitor(allow_auth_retry=True):
    global unity_zrok_process
    unity_zrok_process = start_reserved_zrok()
    public_url = None
    saw_unauthorized = False

    for _ in range(30):
        line = unity_zrok_process.stdout.readline()
        if line:
            line = line.strip()
            print(line)

            if is_unauthorized(line):
                saw_unauthorized = True
                break

            try:
                payload = json.loads(line)
                msg = payload.get("msg", "")
                public_url = extract_url(msg) or extract_url(line)
            except json.JSONDecodeError:
                public_url = extract_url(line)

            if public_url:
                return public_url, False

        time.sleep(1)

    if saw_unauthorized and allow_auth_retry:
        print("⚠️ zrok share reported unauthorized. Re-enabling and retrying once...")
        if ensure_zrok_auth(force=True):
            return start_and_monitor(allow_auth_retry=False)

    return None, saw_unauthorized


stop_existing_unity_tunnel()
reserve_zrok()
print(f"🔒 Starting reserved Unity zrok share '{RESERVED_NAME}' for {TARGET}...")
public_url, _ = start_and_monitor(allow_auth_retry=True)

if not public_url:
    print("❌ Failed to start reserved zrok tunnel. Confirm the reserved share exists in zrok and points to http://127.0.0.1:8000.")
else:
    print(f"\n🚀 PUBLIC ENDPOINT: {public_url}\n")
    if public_url.rstrip("/") != RESERVED_ENDPOINT:
        print(f"ℹ️ Expected reserved endpoint: {RESERVED_ENDPOINT}")

♻️ Stopping previous Unity zrok tunnel process from this notebook...
🔄 Ensuring 'comfyuitunnel' is reserved...
ℹ️ Reserve command output (expected if already reserved):  [ERROR]: unable to create share (unable to create share: [POST /share][409] shareConflict)
🔒 Starting reserved Unity zrok share 'comfyuitunnel' for http://127.0.0.1:8000...
{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:132","func":"main.(*shareReservedCommand).shareLocal","level":"info","msg":"sharing target: 'http://127.0.0.1:8000'","time":"2026-06-01T14:51:50.251Z"}
{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:147","func":"main.(*shareReservedCommand).shareLocal","level":"info","msg":"using existing backend target: http://127.0.0.1:8000","time":"2026-06-01T14:51:50.251Z"}
{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:330","func":"main.(*shareReservedCommand).shareLocal","level":"info","msg":"access your zrok share: https://comfyuitunnel.share.zrok.io","time":"2026-06-01T14:51:50.791Z"}

🚀 PUBLIC ENDPOINT

In [67]:
import requests
print(requests.get("http://127.0.0.1:8188/system_stats", timeout=10).status_code)
print(requests.get("http://127.0.0.1:8000/health", timeout=10).text)

200
INFO:     127.0.0.1:35650 - "GET /health HTTP/1.1" 200 OK
{"status":"ok","comfyui":"reachable"}


In [ ]:
# Live ComfyUI log tail until Unity generation completes/errors.
import os
import time
import requests

LOG_PATH = "/tmp/spatialgen_comfyui.log"
UNITY_STATUS_URL = "http://127.0.0.1:8000/unity_status"

TAIL_UNTIL_DONE = True
TAIL_SECONDS = 1800
TAIL_INTERVAL_SECONDS = 2

if TAIL_UNTIL_DONE:
    deadline = time.time() + TAIL_SECONDS
    log_pos = 0

    if os.path.exists(LOG_PATH):
        log_pos = max(0, os.path.getsize(LOG_PATH) - 8000)

    while time.time() < deadline:
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", encoding="utf-8", errors="replace") as f:
                f.seek(log_pos)
                chunk = f.read()
                log_pos = f.tell()
            if chunk:
                print(chunk, end="")

        try:
            payload = requests.get(UNITY_STATUS_URL, timeout=5).json()
            latest = payload.get("latest") or {}
            status = latest.get("status")
            if latest:
                print(
                    f"\n[unity] status={status} "
                    f"queue={latest.get('queue_state')} "
                    f"elapsed={latest.get('elapsed_seconds')} "
                    f"outputs={latest.get('output_file_count', 0)} "
                    f"meshes={latest.get('mesh_count', 0)}"
                )
            if status in {"completed", "error"}:
                break
        except Exception as exc:
            print(f"\n[unity] status check failed: {exc}")

        time.sleep(TAIL_INTERVAL_SECONDS)
else:
    print("Log tail disabled.")

Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']}
Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'gemv_awq_w4a16', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'quantize_svdquant_w4a4', 'scaled_mm_nvfp4', 'scaled_mm_svdquant_w4a4', 'stochastic_rounding_fp8']}
Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_mxfp8', 'dequantize_nvfp4', 'dequantize_per_tensor